# Experiment: Inspect ERA5 Monthly GRIB Files

Objective:
- Inspect the contents of the ERA5 monthly files.
- Confirm archive members, sample GRIB messages, variables, levels, and basic metadata.


In [ ]:
# Setup: imports and paths
from __future__ import annotations

from pathlib import Path
from tempfile import NamedTemporaryFile
from zipfile import ZipFile, is_zipfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from eccodes import codes_get, codes_grib_new_from_file, codes_release

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

FILE_1 = Path("/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/era5-monthly/uvw_1980-2025_1000hpa-100hpa.grib")
FILE_2 = Path("/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/era5-monthly/geopot_sh_temp_vort_1980-2025_1000hpa-100hpa.grib")

SAMPLE_BYTES = 20_000_000

[FILE_1.exists(), FILE_2.exists()]


## Plan

- Hypothesis: both files are ZIP-wrapped GRIB archives containing monthly ERA5 fields on a shared lat/lon grid.
- Variables to sweep: archive member name, sample size, message count, `shortName`, `typeOfLevel`, `level`, and time keys.
- Metrics to record: parsed message count, unique variables, unique levels, and first-message metadata.


In [ ]:
# Helpers for archive and GRIB inspection
def build_sample(path: Path, sample_bytes: int = SAMPLE_BYTES) -> tuple[Path, dict]:
    info = {
        "file": path.name,
        "exists": path.exists(),
        "path": str(path),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else None,
        "is_zip": is_zipfile(path) if path.exists() else False,
    }

    tmp = NamedTemporaryFile(suffix=".grib", delete=False)
    try:
        if info["is_zip"]:
            with ZipFile(path) as zf:
                member = zf.namelist()[0]
                member_info = zf.getinfo(member)
                info["member"] = member
                info["member_uncompressed_mb"] = round(member_info.file_size / (1024 * 1024), 2)
                with zf.open(member) as src:
                    tmp.write(src.read(sample_bytes))
        else:
            info["member"] = None
            info["member_uncompressed_mb"] = None
            with path.open("rb") as src:
                tmp.write(src.read(sample_bytes))
        tmp.flush()
        return Path(tmp.name), info
    except Exception:
        Path(tmp.name).unlink(missing_ok=True)
        raise
    finally:
        try:
            tmp.close()
        except Exception:
            pass


def parse_grib_messages(sample_path: Path, max_messages: int | None = None) -> pd.DataFrame:
    records = []
    with sample_path.open("rb") as f:
        while True:
            try:
                handle = codes_grib_new_from_file(f)
            except Exception:
                break
            if handle is None:
                break
            record = {}
            for key in ["shortName", "name", "typeOfLevel", "level", "date", "time", "step", "Nx", "Ny"]:
                try:
                    record[key] = codes_get(handle, key)
                except Exception:
                    record[key] = None
            records.append(record)
            codes_release(handle)
            if max_messages is not None and len(records) >= max_messages:
                break
    return pd.DataFrame(records)


def inspect_file(path: Path) -> tuple[dict, pd.DataFrame]:
    sample_path, info = build_sample(path)
    try:
        messages = parse_grib_messages(sample_path)
    finally:
        sample_path.unlink(missing_ok=True)

    summary = {
        **info,
        "sample_bytes": SAMPLE_BYTES,
        "parsed_messages": int(len(messages)),
        "unique_shortNames": sorted(messages["shortName"].dropna().astype(str).unique().tolist()) if not messages.empty else [],
        "unique_levels": sorted(messages["level"].dropna().astype(int).unique().tolist()) if not messages.empty else [],
        "unique_typeOfLevel": sorted(messages["typeOfLevel"].dropna().astype(str).unique().tolist()) if not messages.empty else [],
        "first_message": messages.iloc[0].to_dict() if not messages.empty else {},
    }
    if not messages.empty:
        summary["last_message"] = messages.iloc[-1].to_dict()
    else:
        summary["last_message"] = {}
    return summary, messages


summaries = []
message_tables = {}
for path in [FILE_1, FILE_2]:
    summary, messages = inspect_file(path)
    summaries.append(summary)
    message_tables[path.name] = messages

pd.DataFrame(summaries)


## File Details

The table below shows whether each file is ZIP-wrapped and how many GRIB messages were visible in the sampled chunk.


In [ ]:
details = pd.DataFrame(summaries)[
    [
        "file",
        "size_mb",
        "is_zip",
        "member",
        "member_uncompressed_mb",
        "sample_bytes",
        "parsed_messages",
        "unique_shortNames",
        "unique_levels",
        "unique_typeOfLevel",
    ]
]
details


## Variable Inspection

Display the first sampled GRIB messages for each file. This is enough to confirm variable names, level coverage, and the grid shape without extracting the full payload.


In [ ]:
for name, messages in message_tables.items():
    print("=" * 100)
    print(name)
    print(messages)
    print()
    if not messages.empty:
        display(messages.head(12))


## Quick Plot

A compact bar chart of the sampled GRIB message counts by variable, so you can see the variable mix at a glance.


In [ ]:
for name, messages in message_tables.items():
    if messages.empty:
        continue
    counts = messages["shortName"].value_counts().sort_index()
    ax = counts.plot(kind="bar", color="#2c7fb8")
    ax.set_title(f"Sampled GRIB messages in {name}")
    ax.set_xlabel("shortName")
    ax.set_ylabel("message count")
    plt.tight_layout()
    plt.show()


## Results

- Key observations: note the GRIB variables and the number of levels present in the sampled chunk.
- Surprises or failure modes: record whether the files are ZIP archives and whether the sample had to stop early.
- Decision: continue with downstream analysis once the variable mapping looks correct.


In [ ]:
# Record the main inspection outputs in a compact structure.
result = {
    row["file"]: {
        "parsed_messages": row["parsed_messages"],
        "unique_shortNames": row["unique_shortNames"],
        "unique_levels": row["unique_levels"],
        "unique_typeOfLevel": row["unique_typeOfLevel"],
    }
    for row in summaries
}
result


## Next steps

- What to try next: if you need the full contents, use ecCodes or cdo on the underlying `data.grib` member rather than extracting the whole archive.
- What to document elsewhere (PRD, notes, issue): record the file pairing and the sampled variable/level mapping before building derived datasets.
